In [11]:
from bs4 import BeautifulSoup
import re
import requests

In [12]:
# Writing Example Code to GET a Security Data
security = "SUNPHARMA"
# security = "MARUTI"

url = "https://www.screener.in/company/{}/consolidated/".format(security)

page_source_code = requests.get(url).content

In [13]:
soup = BeautifulSoup(page_source_code, 'html.parser')

In [14]:
table_header = str(soup.find(id = "quarters").thead)

table_header_pattern = r'''<thead>
<tr>
<th class=\"text\"></th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
</tr>
</thead>'''

print(table_header)

<thead>
<tr>
<th class="text"></th>
<th class="highlight-cell">
            Dec 2021
            
          </th>
<th class="">
            Mar 2022
            
          </th>
<th class="">
            Jun 2022
            
          </th>
<th class="">
            Sep 2022
            
          </th>
<th class="highlight-cell">
            Dec 2022
            
          </th>
<th class="">
            Mar 2023
            
          </th>
<th class="">
            Jun 2023
            
          </th>
<th class="">
            Sep 2023
            
          </th>
<th class="highlight-cell">
            Dec 2023
            
          </th>
<th class="">
            Mar 2024
            
          </th>
<th class="">
            Jun 2024
            
          </th>
<th class="">
            Sep 2024
            
          </th>
<th class="highlight-cell">
            Dec 2024
            
          </th>
</tr>
</thead>


In [15]:
match_table_header = re.match(table_header_pattern, table_header)

table_header_month = match_table_header.group(2, 5, 8, 11, 14, 17, 20, 23, 26, 28, 31, 34, 37)
table_header_year = match_table_header.group(3, 6, 9, 12, 15, 18, 21, 24, 27, 29, 32, 35, 38)

print(table_header_month)
print(table_header_year)

('Dec', 'Mar', 'Jun', 'Sep', 'Dec', 'Mar', 'Jun', 'Sep', 'Dec', 'Mar', 'Jun', 'Sep', 'Dec')
('2021', '2022', '2022', '2022', '2022', '2023', '2023', '2023', '2023', '2024', '2024', '2024', '2024')


In [16]:
table_body = soup.find(id = "quarters").tbody.find_all('tr')

# [0] Sales - 0
# [1] Expenses - 1
# [2] PBT - 7
# [3] PAT - 9
# [4] EPS - 10

# Button Pattern Followed by Sales, Expenses and PAT
button_pattern = r'''<tr class=\"(.*)\">
<td class=\"text\">
<button class=\"button-plain\" onclick=\"(.*)\">
                  (.*) <span class=\"blue-icon\">(.*)</span>
</button>
</td>
<td class=\"(.*)\">(.*?)</td>
<td class=\"(.*)\">(.*?)</td>
<td class=\"(.*)\">(.*?)</td>
<td class=\"(.*)\">(.*?)</td>
<td class=\"(.*)\">(.*?)</td>
<td class=\"(.*)\">(.*?)</td>
<td class=\"(.*)\">(.*?)</td>
<td class=\"(.*)\">(.*?)</td>
<td class=\"(.*)\">(.*?)</td>
<td class=\"(.*)\">(.*?)</td>
<td class=\"(.*)\">(.*?)</td>
<td class=\"(.*)\">(.*?)</td>
<td class=\"(.*)\">(.*?)</td>
</tr>'''

# Quarters Data is saved in a single list

quarters = []

for i in range(0, 2):
	expenses_sales = str(table_body[i])    
	match_expenses_sales = re.match(button_pattern, expenses_sales)
	expenses_sales_data = [int(i.replace(',', '')) for i in match_expenses_sales.group(6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30)]
	quarters.append(expenses_sales_data)

In [17]:
profit_before_tax = str(table_body[7])
profit_after_tax = str(table_body[9])
eps = str(table_body[10])

# Non Button Pattern Followed by PBT and EPS
non_button_pattern = r'''<tr class=\"(.*)\">
<td class=\"text\">
              
                (.*)
              
            </td>
<td class=\"(.*)\">(.*?)</td>
<td class=\"\">(.*?)</td>
<td class=\"\">(.*?)</td>
<td class=\"\">(.*?)</td>
<td class=\"(.*)\">(.*?)</td>
<td class=\"\">(.*?)</td>
<td class=\"\">(.*?)</td>
<td class=\"\">(.*?)</td>
<td class=\"(.*)\">(.*?)</td>
<td class=\"\">(.*?)</td>
<td class=\"\">(.*?)</td>
<td class=\"\">(.*?)</td>
<td class=\"(.*)\">(.*?)</td>
</tr>'''

match_pbt = re.match(non_button_pattern, profit_before_tax)
pbt_data = [int(i.replace(',', '')) for i in match_pbt.group(4, 5, 6, 7, 9, 10, 11, 12, 14, 15, 16, 17, 19)]
quarters.append(pbt_data)

match_pat = re.match(button_pattern, profit_after_tax)
pat_data = [int(i.replace(',', '').replace('%', '')) for i in match_pat.group(6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30)]
quarters.append(pat_data)

match_eps = re.match(non_button_pattern, eps)
eps_data = [float(i.replace(',', '')) for i in match_eps.group(4, 5, 6, 7, 9, 10, 11, 12, 14, 15, 16, 17, 19)]
quarters.append(eps_data)

In [18]:
for i in quarters:
    print(i)

[9863, 9447, 10762, 10952, 11241, 10931, 11941, 12192, 12381, 11983, 12653, 13291, 13675]
[7257, 7106, 7877, 7996, 8237, 8129, 8609, 9013, 8904, 8948, 9045, 9352, 9666]
[2466, -2076, 2285, 2412, 2471, 2240, 2481, 2791, 3000, 2816, 3424, 3598, 3476]
[2126, -2227, 2093, 2256, 2181, 1983, 2006, 2385, 2561, 2659, 2861, 3037, 2913]
[8.58, -9.49, 8.59, 9.43, 9.03, 8.27, 8.43, 9.9, 10.52, 11.06, 11.82, 12.67, 12.1]


In [19]:
# Profit & Loss (Yearly)
pl_table_header = str(soup.find(id = "profit-loss").thead)

pl_header_pattern = r'''<thead>
<tr>
<th class=\"text\"></th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?) (.*?)
            
          </th>
<th class=\"(.*)\">
            (.*?)
            
          </th>
</tr>
</thead>'''

match_pl_table_header = re.match(pl_header_pattern, pl_table_header)

pl_table_header_month = match_pl_table_header.group(2, 5, 8, 11, 14, 17, 20, 23, 26, 29, 32, 35, 38)
pl_table_header_year = match_pl_table_header.group(3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33, 36, 38)

print(pl_table_header_month)
print(pl_table_header_year)

('Mar', 'Mar', 'Mar', 'Mar', 'Mar', 'Mar', 'Mar', 'Mar', 'Mar', 'Mar', 'Mar', 'Mar', 'TTM')
('2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', 'TTM')


In [20]:
pl_table_body = soup.find(id = "profit-loss").tbody.find_all('tr')

# [0] Sales - 0
# [1] Expenses - 1
# [2] PBT - 7
# [3] PAT - 9
# [4] EPS - 10

